# Selected ensemble forecast maps

Open one AIFS ensemble forecast case, match it to ERA5 at its verification time, and plot the ensemble mean, spread, observation, and bias. The spatial subset is applied before any data are read.

In [ ]:
import warnings

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from zarr.errors import ZarrUserWarning

import heatextremes as he

variable = "2m_temperature"
region_name = "Delhi, India"

# Choose either an initialization time or an exact verification time.
initialization_time = "2000-08-17T00:00"
target_time = None  # e.g. "2000-08-20T12:00"; infers initialization_time
lead_time = np.timedelta64(3, "D")
target_hour = None  # Optional UTC-hour check: 0, 6, 12, or 18


## Open and subset the inputs

The regional subset is applied to the raw 6-hourly data, before selecting the case.

In [ ]:
era5 = he.open_cached_era5(
    chunks={"time": 244, "latitude": 90, "longitude": 180}
)

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"Numcodecs codecs are not in the Zarr version 3 specification.*",
        category=ZarrUserWarning,
    )
    model = he.open_aifs_ensv2()


In [ ]:
def normalize_longitude(data: xr.Dataset | xr.DataArray) -> xr.Dataset | xr.DataArray:
    """Convert a longitude coordinate from 0--360 to [-180, 180) and sort it."""
    normalized_longitude = (data.longitude + 180) % 360 - 180
    if normalized_longitude.to_index().has_duplicates:
        raise ValueError("Longitude normalization produced duplicate coordinates")
    return data.assign_coords(longitude=normalized_longitude).sortby("longitude")

# Broad rectangular analysis domains: (south, north, west, east).
region_bounds = {
    "Delhi, India": (27.5, 29.5, 76.5, 78.5),
    "Dakar, Senegal": (13.0, 15.5, -18.5, -16.0),
    "Jaipur, India": (25.5, 27.5, 74.5, 77.0),
    "Lagos, Nigeria": (5.0, 8.0, 2.0, 5.0),
    "Addis Ababa, Ethiopia": (7.5, 10.5, 37.5, 40.5),
    "Chicago, IL": (40.0, 43.0, -89.0, -86.0),
    "College Park, MD": (38.0, 40.0, -78.0, -75.5),
    "Bangladesh": (20.0, 27.0, 88.0, 93.0),
    "Europe 2026 heatwave": (35.0, 60.0, -10.0, 35.0),
    "Philippines": (4.0, 22.0, 116.0, 127.0),
    "Senegal": (12.0, 17.5, -18.0, -11.0),
    "Colombia": (-5.0, 13.0, -80.0, -66.0),
    "Rwanda": (-3.0, -1.0, 28.8, 30.9),
    "Pakistan": (23.0, 38.0, 60.0, 78.0),
    "Nigeria": (4.0, 14.5, 2.0, 15.0),
    "Ethiopia": (3.0, 15.0, 32.5, 48.0),
    "Lao PDR": (13.5, 23.0, 100.0, 108.0),
    "Kenya": (-5.0, 5.5, 33.5, 42.0),
    "Peru": (-19.0, 0.5, -82.0, -68.0),
    "Bolivia": (-23.0, -9.0, -70.0, -57.0),
    "Sierra Leone": (6.8, 10.0, -13.5, -10.5),
    "Indonesia": (-11.0, 6.0, 95.0, 141.0),
}

def subset_region(
    data: xr.Dataset | xr.DataArray, bounds: tuple[float, float, float, float]
) -> xr.Dataset | xr.DataArray:
    south, north, west, east = bounds
    latitude_slice = (
        slice(south, north)
        if data.latitude.values[0] < data.latitude.values[-1]
        else slice(north, south)
    )
    return data.sel(latitude=latitude_slice, longitude=slice(west, east))

if region_name not in region_bounds:
    raise KeyError(f"Unknown region: {region_name}. Choose from {list(region_bounds)}")

model = subset_region(normalize_longitude(model), region_bounds[region_name])
era5 = subset_region(normalize_longitude(era5), region_bounds[region_name])
model, era5 = xr.align(
    model,
    era5,
    join="inner",
    exclude={"time", "prediction_timedelta", "number"},
    copy=False,
)


## Select one forecast case

`target_time` is the verification time. When it is supplied, the notebook derives the initialization as `target_time - lead_time`. `target_hour` is optional and verifies that the selected valid time is at the requested UTC hour.

In [ ]:
def select_forecast_case(
    forecast: xr.Dataset,
    variable: str,
    lead_time: np.timedelta64,
    initialization_time: str | None = None,
    target_time: str | None = None,
    target_hour: int | None = None,
) -> tuple[xr.DataArray, np.datetime64]:
    """Return one ensemble case and its valid verification time."""
    if target_time is not None:
        inferred_initialization = np.datetime64(target_time) - lead_time
        if initialization_time is not None and inferred_initialization != np.datetime64(
            initialization_time
        ):
            raise ValueError("initialization_time and target_time disagree for this lead_time")
        initialization = inferred_initialization
    elif initialization_time is not None:
        initialization = np.datetime64(initialization_time)
    else:
        raise ValueError("Set initialization_time or target_time")

    case = forecast[variable].sel(
        time=initialization, prediction_timedelta=lead_time
    )
    verification_time = (case.time + case.prediction_timedelta).values

    if target_time is not None and verification_time != np.datetime64(target_time):
        raise ValueError("Selected forecast does not have the requested target_time")
    if target_hour is not None and int((case.time + case.prediction_timedelta).dt.hour.item()) != target_hour:
        raise ValueError("Selected forecast does not verify at target_hour UTC")

    return case, verification_time

model_case, verification_time = select_forecast_case(
    model,
    variable=variable,
    lead_time=lead_time,
    initialization_time=initialization_time,
    target_time=target_time,
    target_hour=target_hour,
)
era5_case = era5[variable].sel(time=verification_time)

print(f"Initialization: {model_case.time.values}")
print(f"Lead time: {model_case.prediction_timedelta.values}")
print(f"Verification time: {verification_time}")


## Plot maps

Only this selected regional forecast case is computed. Temperatures are shown in degrees Celsius.

In [ ]:
maps = xr.Dataset(
    {
        "ensemble_mean": model_case.mean("number"),
        "ensemble_spread": model_case.std("number"),
        "observation": era5_case,
    }
).compute()
maps["bias"] = maps["ensemble_mean"] - maps["observation"]

ensemble_mean = maps["ensemble_mean"] - 273.15
ensemble_spread = maps["ensemble_spread"]
observation = maps["observation"] - 273.15
bias = maps["bias"]

temperature_min = min(float(ensemble_mean.min()), float(observation.min()))
temperature_max = max(float(ensemble_mean.max()), float(observation.max()))
bias_limit = max(abs(float(bias.min())), abs(float(bias.max())))
south, north, west, east = region_bounds[region_name]

figure, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(12, 9),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True,
)

plot_specs = (
    (ensemble_mean, "Ensemble mean (°C)", "RdYlBu_r", temperature_min, temperature_max),
    (observation, "ERA5 observation (°C)", "RdYlBu_r", temperature_min, temperature_max),
    (ensemble_spread, "Ensemble standard deviation (K)", "magma", 0, None),
    (bias, "Ensemble mean bias (K)", "RdBu_r", -bias_limit, bias_limit),
)

for axis, (field, title, cmap, vmin, vmax) in zip(axes.flat, plot_specs):
    image = field.plot.pcolormesh(
        ax=axis,
        transform=ccrs.PlateCarree(),
        add_colorbar=False,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    axis.set_title(title)
    axis.set_extent([west, east, south, north], crs=ccrs.PlateCarree())
    axis.coastlines()
    figure.colorbar(image, ax=axis, shrink=0.8)

figure.suptitle(
    f"{region_name}: initialization {model_case.time.values}, "
    f"lead {model_case.prediction_timedelta.values}",
    fontsize=14,
)
plt.show()
